# Vitalia HAR — Kaggle (5 clases, sin cycling)

Modelo HAR con 5 clases de smartphone: **static, walking, running, upstairs, downstairs**.
Sin PAMAP2. Solo necesitas el dataset `vitalia-har-processed`.

**Output (`/kaggle/working/`):** `har_model_int8.tflite`, `har_model_fp16.tflite`, `model_meta.json`, `har_model.keras`

In [ ]:
import numpy as np
import pandas as pd
import os, math, shutil, json, warnings, collections
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers as L
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from scipy.signal import resample_poly, butter, filtfilt
from math import gcd

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)
tf.keras.mixed_precision.set_global_policy('mixed_float16')

WORKING_DIR = Path('/kaggle/working')
INPUT_DIR   = Path('/kaggle/input')

print(f'TensorFlow {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
LOAD_FROM_PROCESSED = True
RUN_LOSO            = True
QUICK_LOSO_SUBJECTS = None   # None = all; set e.g. 5 to test quickly
BASELINE_F1         = 0.86   # minimum LOSO F1 required to export TFLite

PROC_DIR        = Path('/kaggle/input/datasets/jaimenovillo/vitalia-har-processed')
UCI_HAR_DIR     = INPUT_DIR / 'uci-har-dataset'
MOTIONSENSE_DIR = INPUT_DIR / 'motionsense'

print(f'PROC_DIR exists: {PROC_DIR.exists()}')

In [ ]:
# ── Preprocessing functions ───────────────────────────────────────────────────
def resample_signal(data, orig_hz, target_hz=50):
    if orig_hz == target_hz: return data
    g = gcd(orig_hz, target_hz)
    return resample_poly(data, target_hz//g, orig_hz//g, axis=0).astype(np.float32)

def sliding_window(data, window_size=128, overlap=0.5):
    step = int(window_size * (1.0 - overlap))
    starts = range(0, len(data) - window_size + 1, step)
    return np.stack([data[s:s+window_size] for s in starts], axis=0).astype(np.float32)

def augment_axis_permutation(w):
    p = np.random.permutation(3); s = np.random.choice([-1,1], size=3)
    aug = w.copy(); aug[:,:3] = w[:,:3][:,p]*s; aug[:,3:6] = w[:,3:6][:,p]*s
    return aug.astype(np.float32)

def augment_gaussian_noise(w, sigma=0.01):
    return (w + np.random.normal(0, sigma, w.shape)).astype(np.float32)

def augment_magnitude_scale(w, lo=0.8, hi=1.2):
    return (w * np.random.uniform(lo, hi)).astype(np.float32)

def augment_time_warp(w, ratio=0.1):
    n, c = w.shape; factor = 1.0 + np.random.uniform(-ratio, ratio)
    new_len = max(1, int(n*factor))
    out = resample_poly(resample_poly(w, new_len, n, axis=0), n, new_len, axis=0)
    if len(out) >= n: return out[:n].astype(np.float32)
    return np.concatenate([out, np.zeros((n-len(out), c), dtype=np.float32)])

def augment_window(w, n_augments=4):
    out = []
    for _ in range(n_augments):
        aug = w.copy()
        if np.random.rand() > 0.5: aug = augment_axis_permutation(aug)
        if np.random.rand() > 0.5: aug = augment_gaussian_noise(aug)
        if np.random.rand() > 0.5: aug = augment_magnitude_scale(aug)
        if np.random.rand() > 0.5: aug = augment_time_warp(aug)
        out.append(aug)
    return out

print('Preprocessing functions ready.')

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────────
# Canonical 5-class order (must match har_classifier.dart Activity enum):
#   0=static  1=walking  2=running  3=upstairs  4=downstairs
# Old UCI/MotionSense labels (0-indexed): 0=walking,1=upstairs,2=downstairs,3=sitting,4=standing,5=running
_REMAP = {0: 1, 1: 3, 2: 4, 3: 0, 4: 0, 5: 2}  # sitting+standing → static(0)

if LOAD_FROM_PROCESSED and PROC_DIR.exists():
    print('Loading preprocessed .npy arrays...')
    X        = np.load(PROC_DIR / 'X_activities.npy')
    y_raw    = np.load(PROC_DIR / 'y_activities.npy')
    subjects = np.load(PROC_DIR / 'subjects_activities.npy')
    y_0 = np.vectorize(_REMAP.get)(y_raw - 1).astype(np.int32)
    print(f'  X={X.shape}, subjects={len(np.unique(subjects))}')
else:
    raise RuntimeError('Add dataset vitalia-har-processed, or set LOAD_FROM_PROCESSED=False and add uci-har-dataset + motionsense')

N_CLASSES  = 5
CLASS_NAMES = ['static', 'walking', 'running', 'upstairs', 'downstairs']

print(f'N_CLASSES={N_CLASSES}  CLASS_NAMES={CLASS_NAMES}')
print('Class distribution:', dict(sorted(collections.Counter(y_0.tolist()).items())))

In [ ]:
# ── ResNet1D v2 ────────────────────────────────────────────────────────────────
def build_har_model(n_classes=5, window_size=128, n_channels=6):
    inp = keras.Input(shape=(window_size, n_channels), name='sensor_input')
    x = L.Conv1D(64, 7, padding='same', use_bias=False, name='stem_conv')(inp)
    x = L.BatchNormalization(name='stem_bn')(x)
    x = L.Activation('relu', name='stem_act')(x)
    sc = x
    x = L.Conv1D(64, 3, padding='same', use_bias=False, name='r1_c1')(x)
    x = L.BatchNormalization(name='r1_bn1')(x)
    x = L.Activation('relu', name='r1_a1')(x)
    x = L.Conv1D(64, 3, padding='same', use_bias=False, name='r1_c2')(x)
    x = L.BatchNormalization(name='r1_bn2')(x)
    x = L.Add(name='r1_add')([x, sc])
    x = L.Activation('relu', name='r1_out')(x)
    x = L.MaxPooling1D(2, name='pool1')(x)
    sc = L.Conv1D(128, 1, padding='same', use_bias=False, name='r2_proj')(x)
    x = L.Conv1D(128, 3, padding='same', use_bias=False, name='r2_c1')(x)
    x = L.BatchNormalization(name='r2_bn1')(x)
    x = L.Activation('relu', name='r2_a1')(x)
    x = L.Conv1D(128, 3, padding='same', use_bias=False, name='r2_c2')(x)
    x = L.BatchNormalization(name='r2_bn2')(x)
    x = L.Add(name='r2_add')([x, sc])
    x = L.Activation('relu', name='r2_out')(x)
    x = L.MaxPooling1D(2, name='pool2')(x)
    sc = L.Conv1D(256, 1, padding='same', use_bias=False, name='r3_proj')(x)
    x = L.Conv1D(256, 3, padding='same', use_bias=False, name='r3_c1')(x)
    x = L.BatchNormalization(name='r3_bn1')(x)
    x = L.Activation('relu', name='r3_a1')(x)
    x = L.Conv1D(256, 3, padding='same', use_bias=False, name='r3_c2')(x)
    x = L.BatchNormalization(name='r3_bn2')(x)
    x = L.Add(name='r3_add')([x, sc])
    x = L.Activation('relu', name='r3_out')(x)
    x = L.GlobalAveragePooling1D(name='gap')(x)
    x = L.Dense(128, activation='relu', name='fc1', dtype='float32')(x)
    x = L.Dropout(0.4, name='drop')(x)
    out = L.Dense(n_classes, activation='softmax', name='class_probs', dtype='float32')(x)
    return keras.Model(inp, out, name='HAR_ResNet1D_v2')

_m = build_har_model(); _m.summary(); del _m

In [ ]:
# ── Training helpers ───────────────────────────────────────────────────────────
# 0=static,1=walking,2=running,3=upstairs,4=downstairs
AUG_MULTIPLIERS = {0: 1, 1: 1, 2: 5, 3: 2, 4: 2}
CLASS_WEIGHT    = {0: 1.0, 1: 1.0, 2: 3.5, 3: 1.8, 4: 2.1}

def augment_per_class(X_tr, y_tr):
    X_parts, y_parts = [X_tr], [y_tr]
    for cls, mult in AUG_MULTIPLIERS.items():
        if mult <= 1: continue
        idx = np.where(y_tr == cls)[0]
        if len(idx) == 0: continue
        for _ in range(mult - 1):
            wins = [augment_window(X_tr[i], n_augments=1)[0] for i in idx]
            X_parts.append(np.stack(wins))
            y_parts.append(np.full(len(idx), cls, dtype=y_tr.dtype))
    return np.concatenate(X_parts), np.concatenate(y_parts)

def cosine_lr(epoch, total=80, warmup=5, lr_max=1e-3, lr_min=1e-5):
    if epoch < warmup: return lr_max * (epoch+1) / warmup
    t = (epoch - warmup) / (total - warmup)
    return float(lr_min + 0.5*(lr_max-lr_min)*(1+math.cos(math.pi*t)))

print(f'AUG_MULTIPLIERS: {AUG_MULTIPLIERS}')

In [ ]:
# ── LOSO training ──────────────────────────────────────────────────────────────
all_y_true, all_y_pred = [], []
loso_results = []
LOSO_N_SUBJECTS = 0

if RUN_LOSO:
    unique_subj = np.unique(subjects)
    if QUICK_LOSO_SUBJECTS:
        unique_subj = unique_subj[:QUICK_LOSO_SUBJECTS]
    print(f'LOSO over {len(unique_subj)} subjects...')

    for fold_i, test_subj in enumerate(unique_subj):
        test_mask  = subjects == test_subj
        train_mask = ~test_mask
        X_tr, y_tr = X[train_mask], y_0[train_mask]
        X_te, y_te = X[test_mask],  y_0[test_mask]
        X_tr, y_tr = augment_per_class(X_tr, y_tr)
        X_tr, X_val, y_tr, y_val = train_test_split(X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=42)

        m = build_har_model(N_CLASSES)
        m.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        m.fit(X_tr, y_tr, epochs=80, batch_size=128,
              validation_data=(X_val, y_val), class_weight=CLASS_WEIGHT,
              callbacks=[
                  keras.callbacks.LearningRateScheduler(cosine_lr, verbose=0),
                  keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=0),
              ], verbose=0)

        y_pred = m.predict(X_te, verbose=0).argmax(axis=1)
        all_y_true.extend(y_te); all_y_pred.extend(y_pred)
        f1 = f1_score(y_te, y_pred, average='macro')
        acc = accuracy_score(y_te, y_pred)
        loso_results.append({'subject': int(test_subj), 'f1_macro': f1, 'accuracy': acc})
        print(f'  [{fold_i+1}/{len(unique_subj)}] subj {test_subj}: F1={f1:.3f} acc={acc:.3f}')
        del m

    all_y_true = np.array(all_y_true)
    all_y_pred = np.array(all_y_pred)
    LOSO_N_SUBJECTS = len(unique_subj)
    loso_f1 = f1_score(all_y_true, all_y_pred, average='macro')

    print(f'\nLOSO F1-macro: {loso_f1:.4f}')
    print(classification_report(all_y_true, all_y_pred,
          target_names=CLASS_NAMES, labels=list(range(N_CLASSES)), zero_division=0))
    pd.DataFrame(loso_results).to_csv(WORKING_DIR / 'loso_results.csv', index=False)
else:
    print('LOSO skipped.')
    loso_f1 = None

In [ ]:
# ── Confusion matrix ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt, seaborn as sns

if RUN_LOSO and len(all_y_true) > 0:
    cm = confusion_matrix(all_y_true, all_y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f'ResNet1D v2 — LOSO ({LOSO_N_SUBJECTS} subjects, no cycling)')
    plt.tight_layout()
    plt.savefig(WORKING_DIR / 'confusion_matrix.png', dpi=120)
    plt.show()

    f1_per_class = f1_score(all_y_true, all_y_pred, average=None,
                            labels=list(range(N_CLASSES)), zero_division=0)
    fig, ax = plt.subplots(figsize=(7, 3))
    colors = ['red' if f < 0.85 else 'steelblue' for f in f1_per_class]
    ax.bar(CLASS_NAMES, f1_per_class, color=colors)
    ax.axhline(0.85, color='red', linestyle='--', alpha=0.5, label='target 0.85')
    ax.set_ylim(0, 1.05); ax.set_ylabel('F1'); ax.legend()
    plt.tight_layout()
    plt.savefig(WORKING_DIR / 'f1_per_class.png', dpi=120)
    plt.show()

In [ ]:
# ── Final model — train on ALL data ───────────────────────────────────────────
print('Training final model on full dataset...')
X_full, y_full = augment_per_class(X, y_0)
unique_cls, cls_counts = np.unique(y_full, return_counts=True)
for cls, cnt in zip(unique_cls, cls_counts):
    print(f'  {CLASS_NAMES[cls]:12s}: {cnt:6d}')

X_full, X_val, y_full, y_val = train_test_split(
    X_full, y_full, test_size=0.05, stratify=y_full, random_state=42)
print(f'Train: {len(X_full)}  Val: {len(X_val)}')

final_model = build_har_model(N_CLASSES)
final_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])
final_model.fit(
    X_full, y_full, epochs=80, batch_size=128,
    validation_data=(X_val, y_val), class_weight=CLASS_WEIGHT,
    callbacks=[
        keras.callbacks.LearningRateScheduler(cosine_lr, verbose=0),
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint(
            str(WORKING_DIR / 'har_model.keras'),
            save_best_only=True, monitor='val_accuracy', verbose=1),
    ], verbose=1)
print('Saved to /kaggle/working/har_model.keras')

In [ ]:
# ── TFLite export ──────────────────────────────────────────────────────────────
proceed = (loso_f1 is None) or (loso_f1 >= BASELINE_F1)
if not proceed:
    print(f'WARNING: LOSO F1 {loso_f1:.4f} < {BASELINE_F1}. Not exporting.')
else:
    # Rebuild in float32 (mixed_float16 breaks INT8 conversion)
    tf.keras.mixed_precision.set_global_policy('float32')
    float32_weights = [w.astype(np.float32) for w in final_model.get_weights()]
    final_model = build_har_model(N_CLASSES)
    final_model.set_weights(float32_weights)
    print('Model rebuilt in float32.')

    # INT8
    converter = tf.lite.TFLiteConverter.from_keras_model(final_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    def representative_dataset():
        for i in np.random.choice(len(X), size=min(400, len(X)), replace=False):
            yield [X[i:i+1].astype(np.float32)]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type  = tf.int8
    converter.inference_output_type = tf.int8
    tflite_int8 = converter.convert()
    (WORKING_DIR / 'har_model_int8.tflite').write_bytes(tflite_int8)
    print(f'INT8: {len(tflite_int8)/1024:.1f} KB')

    # FP16
    converter2 = tf.lite.TFLiteConverter.from_keras_model(final_model)
    converter2.optimizations = [tf.lite.Optimize.DEFAULT]
    converter2.target_spec.supported_types = [tf.float16]
    tflite_fp16 = converter2.convert()
    (WORKING_DIR / 'har_model_fp16.tflite').write_bytes(tflite_fp16)
    print(f'FP16: {len(tflite_fp16)/1024:.1f} KB')

    interp = tf.lite.Interpreter(model_content=tflite_int8)
    interp.allocate_tensors()
    out_det = interp.get_output_details()[0]
    assert out_det['shape'][1] == N_CLASSES
    print(f'Output shape: {out_det["shape"]} OK')

In [ ]:
# ── Save metadata ──────────────────────────────────────────────────────────────
meta = {
    'class_names': CLASS_NAMES,
    'n_classes': N_CLASSES,
    'window_size': 128, 'overlap': 0.5, 'target_hz': 50,
    'channels': ['lax','lay','laz','gx','gy','gz'],
    'cycling_trained': False,
    'loso_f1_macro': round(float(loso_f1), 4) if loso_f1 else None,
    'loso_n_subjects': LOSO_N_SUBJECTS,
    'training_windows': int(len(X)),
    'vitapoints_per_min': {'static':0,'walking':2,'running':5,'upstairs':3,'downstairs':2},
}
if proceed:
    meta['int8_size_kb'] = round(len(tflite_int8)/1024, 1)
    meta['fp16_size_kb'] = round(len(tflite_fp16)/1024, 1)

(WORKING_DIR / 'model_meta.json').write_text(json.dumps(meta, indent=2))

print('\n=== VITALIA HAR (5 classes, no cycling) ===')
print(f'Classes: {", ".join(CLASS_NAMES)}')
print(f'LOSO F1: {loso_f1:.4f} ({LOSO_N_SUBJECTS} subjects)' if loso_f1 else 'LOSO: skipped')
print(f'TFLite:  {proceed}')
print('\nFiles:')
for f in sorted(WORKING_DIR.iterdir()):
    print(f'  {f.name} ({f.stat().st_size/1024:.1f} KB)')

## Usar en Flutter

```bash
cp har_model_int8.tflite app/assets/models/har_model_int8.tflite
```

**IMPORTANTE:** El enum `Activity` en `har_classifier.dart` debe tener exactamente 5 valores en este orden:
```dart
enum Activity {
  stationary, // 0
  walking,    // 1
  running,    // 2
  upstairs,   // 3
  downstairs, // 4
}
```
Si el modelo actual tiene 6 clases (con cycling), actualiza el enum antes de usar este modelo.